In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

import config as cnf 
import utils as utl

In [44]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

X, y = utl.predobr(df_train, df_test, df_gender_submission)

combo_param = utl.parametrs(cnf.RandomForestParam) + utl.parametrs(cnf.LogRegParam)
combo_param += utl.parametrs(cnf.KNNParam)
combo_param += utl.parametrs(cnf.DecisionTreeParam)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)
results = []

for combo in combo_param:
    
    params = combo.copy()
    name_model = params.pop('name_model')
    scaler = params.pop('scaler')
    
    print("Model: ", name_model, "Params: ", params)
    
    model = cnf.models[name_model](**params)
    mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.skf_function(model, X, y, skf, scaler)

    results.append({
        **combo
        , 'train_accuracy'  :   mean_train_accuracy
        , 'train_std'       :   mean_train_std
        , 'test_accuracy'   :   mean_test_accuracy
        , 'test_std'        :   mean_test_std
    })

Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.9124576484863767 Test_accuracy:  0.8383340656581508
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.890290275304537 Test_accuracy:  0.8316301550436258
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.879347432119388 Test_accuracy:  0.8293766869625259
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.9295719936334841 Test_accuracy:  0.828253091456908
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.8992672203224231 Test_accuracy:  0.831617600903898
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 3, 'n_estimators': 100, 'ra

### Вывод

In [42]:
results = pd.DataFrame(results)
results['gap'] = results['train_accuracy'] - results['test_accuracy']

# Порядок колонок: сначала модель, потом метрики, потом гиперпараметры
metrics = ['test_accuracy', 'test_std', 'train_accuracy', 'train_std', 'gap']
service = ['name_model', 'scaler', 'random_state']
params = [c for c in results.columns if c not in metrics + service]

results = results[['name_model'] + metrics + params + ['scaler', 'random_state']]

# Сортировка: лучшие по test_accuracy сверху
results = results.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
results.head(50)

,name_model,test_accuracy,test_std,train_accuracy,train_std,gap,max_depth,min_samples_leaf,n_estimators,n_neighbors,scaler,random_state
0,RandomForest,0.842854,0.012525,0.932659,0.004454,0.089805,9.0,1.0,100.0,NaN,False,42.0
1,RandomForest,0.839483,0.019818,0.888046,0.004415,0.048563,9.0,3.0,100.0,NaN,False,42.0
2,RandomForest,0.837236,0.022066,0.894780,0.004647,0.057544,8.0,2.0,100.0,NaN,False,42.0
3,RandomForest,0.836125,0.014097,0.915823,0.006486,0.079698,8.0,1.0,100.0,NaN,False,42.0
4,RandomForest,0.834995,0.020443,0.883558,0.005809,0.048563,8.0,3.0,100.0,NaN,False,42.0
5,RandomForest,0.832754,0.018059,0.905162,0.002320,0.072408,9.0,2.0,100.0,NaN,False,42.0
6,DecisionTree,0.829383,0.015938,0.879064,0.008909,0.049681,8.0,3.0,NaN,NaN,False,42.0
7,DecisionTree,0.827142,0.018064,0.887763,0.006209,0.060621,9.0,3.0,NaN,NaN,False,42.0
8,DecisionTree,0.823784,0.012735,0.883555,0.006451,0.059771,8.0,2.0,NaN,NaN,False,42.0
9,DecisionTree,0.821549,0.020218,0.902074,0.005327,0.080525,8.0,1.0,NaN,NaN,False,42.0
